# Bootstrap the client with ROOT credentials
Using the python client generated from our OpenAPI spec, we generate a token from our root user's credentials

In [1]:
from polaris.catalog.api.iceberg_catalog_api import IcebergCatalogAPI
from polaris.catalog.api.iceberg_o_auth2_api import IcebergOAuth2API
from polaris.catalog.api_client import ApiClient as CatalogApiClient
from polaris.catalog.api_client import Configuration as CatalogApiClientConfiguration

polaris_credential = 'root:s3cr3t' # pragma: allowlist secret

client_id, client_secret = polaris_credential.split(":")
client = CatalogApiClient(CatalogApiClientConfiguration(username=client_id,
                                 password=client_secret,
                                 host='http://polaris:8181/api/catalog'))

oauth_api = IcebergOAuth2API(client)
token = oauth_api.get_token(scope='PRINCIPAL_ROLE:ALL',
                            client_id=client_id,
                          client_secret=client_secret,
                          grant_type='client_credentials',
                          _headers={'realm': 'default-realm'})


# Create our first catalog

* Creates a catalog named `polaris_catalog` that writes to a specified location in the Local Filesystem.

In [2]:
from polaris.management import *

client = ApiClient(Configuration(access_token=token.access_token,
                                   host='http://polaris:8181/api/management/v1'))
root_client = PolarisDefaultApi(client)

storage_conf = FileStorageConfigInfo(storage_type="FILE", allowed_locations=["file:///tmp"])
catalog_name = 'polaris_demo'
catalog = Catalog(name=catalog_name, type='INTERNAL', properties={"default-base-location": "file:///tmp/polaris/"},
                storage_config_info=storage_conf)
catalog.storage_config_info = storage_conf
root_client.create_catalog(create_catalog_request=CreateCatalogRequest(catalog=catalog))
resp = root_client.get_catalog(catalog_name=catalog.name)
resp

PolarisCatalog(type='INTERNAL', name='polaris_demo', properties=CatalogProperties(default_base_location='file:///tmp/polaris/', additional_properties={}), create_timestamp=1739832036425, last_update_timestamp=1739832036425, entity_version=1, storage_config_info=FileStorageConfigInfo(storage_type='FILE', allowed_locations=['file:///tmp', 'file:///tmp/polaris/']))

# Utility Functions

In [3]:
# Creates a principal with the given name
def create_principal(api, principal_name):
  principal = Principal(name=principal_name, type="SERVICE")
  try:
    principal_result = api.create_principal(CreatePrincipalRequest(principal=principal))
    return principal_result
  except ApiException as e:
    if e.status == 409:
      return api.rotate_credentials(principal_name=principal_name)
    else:
      raise e

# Create a catalog role with the given name
def create_catalog_role(api, catalog, role_name):
  catalog_role = CatalogRole(name=role_name)
  try:
    api.create_catalog_role(catalog_name=catalog.name, create_catalog_role_request=CreateCatalogRoleRequest(catalog_role=catalog_role))
    return api.get_catalog_role(catalog_name=catalog.name, catalog_role_name=role_name)
  except ApiException as e:
    return api.get_catalog_role(catalog_name=catalog.name, catalog_role_name=role_name)
  else:
    raise e

# Create a principal role with the given name
def create_principal_role(api, role_name):
  principal_role = PrincipalRole(name=role_name)
  try:
    api.create_principal_role(CreatePrincipalRoleRequest(principal_role=principal_role))
    return api.get_principal_role(principal_role_name=role_name)
  except ApiException as e:
    return api.get_principal_role(principal_role_name=role_name)


# Create a new Principal, Principal Role, and Catalog Role
The new Principal belongs to the `engineer` principal role, which has `CATALOG_MANAGE_CONTENT` privileges on the `polaris_catalog`. 


`CATALOG_MANAGE_CONTENT` has create/list/read/write privileges on all entities within the catalog. The same privilege could be granted to a namespace, in which case, the engineers could create/list/read/write any entity under that namespace

In [4]:
# Create the engineer_principal
engineer_principal = create_principal(root_client, "collado")

# Create the principal role
engineer_role = create_principal_role(root_client, "engineer")

# Create the catalog role
manager_catalog_role = create_catalog_role(root_client, catalog, "manage_catalog")

# Grant the catalog role to the principal role
# All principals in the principal role have the catalog role's privileges
root_client.assign_catalog_role_to_principal_role(principal_role_name=engineer_role.name,
                                                  catalog_name=catalog.name,
                                                  grant_catalog_role_request=GrantCatalogRoleRequest(catalog_role=manager_catalog_role))

# Assign privileges to the catalog role
# Here, we grant CATALOG_MANAGE_CONTENT
root_client.add_grant_to_catalog_role(catalog.name, manager_catalog_role.name,
                                      AddGrantRequest(grant=CatalogGrant(catalog_name=catalog.name,
                                                                       type='catalog',
                                                                       privilege=CatalogPrivilege.CATALOG_MANAGE_CONTENT)))

# Assign the principal role to the principal
root_client.assign_principal_role(engineer_principal.principal.name, grant_principal_role_request=GrantPrincipalRoleRequest(principal_role=engineer_role))

# Create a reader Principal, Principal Role, and Catalog Role
This new principal belongs to the `product_manager` principal role, which is explicitly granted read and list permissions on the catalog.

Permissions cascade, so permissions granted at the catalog level are inherited by namespaces and tables within the catalog.

In [5]:
# Create a reader principal
reader_principal = create_principal(root_client, "mlee")

# Create the principal role
pm_role = create_principal_role(root_client, "product_manager")

# Create the catalog role
read_only_role = create_catalog_role(root_client, catalog, "read_only")

# Grant the catalog role to the principal role
root_client.assign_catalog_role_to_principal_role(principal_role_name=pm_role.name,
                                                  catalog_name=catalog.name,
                                                  grant_catalog_role_request=GrantCatalogRoleRequest(catalog_role=read_only_role))

# Assign privileges to the catalog role
# Here, the catalog role is granted READ and LIST privileges at the catalog level
# Privileges cascade down
root_client.add_grant_to_catalog_role(catalog.name, read_only_role.name,
                                      AddGrantRequest(grant=CatalogGrant(catalog_name=catalog.name,
                                                                       type='catalog',
                                                                       privilege=CatalogPrivilege.TABLE_LIST)))
root_client.add_grant_to_catalog_role(catalog.name, read_only_role.name,
                                      AddGrantRequest(grant=CatalogGrant(catalog_name=catalog.name,
                                                                       type='catalog',
                                                                       privilege=CatalogPrivilege.TABLE_READ_PROPERTIES)))
root_client.add_grant_to_catalog_role(catalog.name, read_only_role.name,
                                      AddGrantRequest(grant=CatalogGrant(catalog_name=catalog.name,
                                                                       type='catalog',
                                                                       privilege=CatalogPrivilege.TABLE_READ_DATA)))
root_client.add_grant_to_catalog_role(catalog.name, read_only_role.name,
                                      AddGrantRequest(grant=CatalogGrant(catalog_name=catalog.name,
                                                                       type='catalog',
                                                                       privilege=CatalogPrivilege.VIEW_LIST)))
root_client.add_grant_to_catalog_role(catalog.name, read_only_role.name,
                                      AddGrantRequest(grant=CatalogGrant(catalog_name=catalog.name,
                                                                       type='catalog',
                                                                       privilege=CatalogPrivilege.VIEW_READ_PROPERTIES)))
root_client.add_grant_to_catalog_role(catalog.name, read_only_role.name,
                                      AddGrantRequest(grant=CatalogGrant(catalog_name=catalog.name,
                                                                       type='catalog',
                                                                       privilege=CatalogPrivilege.NAMESPACE_READ_PROPERTIES)))
root_client.add_grant_to_catalog_role(catalog.name, read_only_role.name,
                                      AddGrantRequest(grant=CatalogGrant(catalog_name=catalog.name,
                                                                       type='catalog',
                                                                       privilege=CatalogPrivilege.NAMESPACE_LIST)))

# Assign the principal role to the principal
root_client.assign_principal_role(reader_principal.principal.name, grant_principal_role_request=GrantPrincipalRoleRequest(principal_role=pm_role))

In [11]:
# PyIceberg Setup instead
from pyiceberg.catalog import load_catalog

pyiceberg_catalog = load_catalog(name=catalog_name, **{
    "type": "rest",
    "header.X-Iceberg-Access-Delegation": "vended-credentials",
    "uri": "http://polaris:8181/api/catalog",
    "credential": f"{engineer_principal.credentials.client_id}:{engineer_principal.credentials.client_secret}",
    "warehouse": catalog_name,
    "token-refresh-enabled": "true",
    "scope": "PRINCIPAL_ROLE:ALL",
    "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO",
}
)

/opt/conda/lib/python3.11/site-packages/pyiceberg/utils/deprecated.py:54: DeprecationWarning: Deprecated in 0.8.0, will be removed in 1.0.0. Iceberg REST client is missing the OAuth2 server URI configuration and defaults to http://polaris:8181/api/catalogoauth/tokens. This automatic fallback will be removed in a future Iceberg release.It is recommended to configure the OAuth2 endpoint using the 'oauth2-server-uri'property to be prepared. This warning will disappear if the OAuth2endpoint is explicitly configured. See https://github.com/apache/iceberg/issues/10537
  _deprecation_warning(deprecation_notice(deprecated_in, removed_in, help_message))


In [16]:
pyiceberg_catalog.create_namespace_if_not_exists("NS1")
pyiceberg_catalog.create_namespace_if_not_exists("NS1.NS2")
pyiceberg_catalog.create_namespace_if_not_exists("NS_POLICY")

In [19]:
# Create some example iceberg tables
import pyarrow as pa
arrow_schema = pa.schema(
    [
        ("foo", pa.bool_()),
        ("bar", pa.string()),
        ("baz", pa.date32()),
    ]
)
pyiceberg_catalog.create_table_if_not_exists(identifier="NS1.NS2.TEST_TABLE_1", schema=arrow_schema)
pyiceberg_catalog.create_table_if_not_exists(identifier="NS1.NS2.TEST_TABLE_2", schema=arrow_schema)

TEST_TABLE_2(
  1: foo: optional boolean,
  2: bar: optional string,
  3: baz: optional date
),
partition by: [],
sort order: [],
snapshot: null

# Use the Catalog API client
Create a new client using the engineer credentials

In [37]:
# Create a client to fetch an API token - use our client_id and client_secret as the username/password
token_client = CatalogApiClient(CatalogApiClientConfiguration(username=engineer_principal.credentials.client_id,
                                 password=engineer_principal.credentials.client_secret,
                                 host='http://polaris:8181/api/catalog'))

# Use the client to get the token from the /tokens endpoint
collado_token = IcebergOAuth2API(token_client).get_token(scope='PRINCIPAL_ROLE:ALL',
                            client_id=engineer_principal.credentials.client_id,
                          client_secret=engineer_principal.credentials.client_secret,
                          grant_type='client_credentials',
                          _headers={'realm': 'default-realm'})

# Now create a catalog client that uses the token in its Authentication header
client = CatalogApiClient(CatalogApiClientConfiguration(access_token=collado_token.access_token,
              host='http://polaris:8181/api/catalog'))
collado_client = IcebergCatalogAPI(client)


In [56]:
# Utilities for policies
import codecs
import json
from IPython.display import display, JSON

from polaris.catalog import CreatePolicyRequest, UpdatePolicyRequest, TableLikeIdentifier, SetPolicyRequest

def format_namespace(namespace):
  return codecs.decode("1F", "hex").decode("UTF-8").join(namespace)


def demo_create_policy(namespace_list, policy_name, policy_type, policy_content, policy_description):
    request = CreatePolicyRequest(name=policy_name, type=policy_type, content=policy_content, description=policy_description)
    return collado_client.create_policy(prefix=catalog_name, namespace=format_namespace(namespace_list), create_policy_request=request)

def demo_load_policy(namespace_list, policy_name):
    return collado_client.get_policy(prefix=catalog_name, namespace=format_namespace(namespace_list), policy_name=policy_name)

def demo_update_policy(namespace_list, policy_name, new_content, new_description):
    update_request = UpdatePolicyRequest(content=new_content, description=new_description)
    return collado_client.update_policy(prefix=catalog_name, namespace=format_namespace(namespace_list), policy_name=policy_name, update_policy_request=update_request)

def demo_drop_policy(namespace_list, policy_name):
    collado_client.delete_policy(prefix=catalog_name, namespace=format_namespace(namespace_list), policy_name=policy_name)

def demo_set_policy_on_table(namespace_list, policy_name, table_namespace_list, table_name):
    table_target = TableLikeIdentifier(type="table-like", catalog=catalog_name, namespace=table_namespace_list, name=table_name)
    request = SetPolicyRequest(entity=table_target, parameters={})
    collado_client.set_policy(prefix=catalog_name, namespace=format_namespace(namespace_list), policy_name=policy_name, set_policy_request=request)

# Demo Hierarchy

```
catalog:
    NS1
        NS2
            TEST_TABLE_1
            TEST_TABLE_1
    NS_POLICY
        POLICY_1
        POLICY_2
        POLICY_3
```



In [58]:
demo_create_policy(namespace_list=["NS_POLICY"], policy_name="POLICY_1", policy_type="test_type_1", policy_content="demo_content", policy_description="description")
demo_create_policy(namespace_list=["NS_POLICY"], policy_name="POLICY_2", policy_type="test_type_2", policy_content="demo_content", policy_description="description")
demo_create_policy(namespace_list=["NS_POLICY"], policy_name="POLICY_3", policy_type="test_type_3", policy_content="demo_content", policy_description="description")

LoadPolicyResult(policy=Policy(policy_type='test_type_3', name='POLICY_3', description='description', content='demo_content', version=0, created_at_ms=1739865884147, updated_at_ms=1739865884147))

In [59]:
load_policy_result = demo_load_policy(["NS_POLICY"], "POLICY_1")
load_policy_result.policy

Policy(policy_type='test_type_1', name='POLICY_1', description='description', content='demo_content', version=0, created_at_ms=1739865884136, updated_at_ms=1739865884136)

In [61]:
demo_update_policy(["NS_POLICY"], "POLICY_1", new_content="new_content", new_description="new_description")

LoadPolicyResult(policy=Policy(policy_type='test_type_1', name='POLICY_1', description='new_description', content='new_content', version=0, created_at_ms=1739865884136, updated_at_ms=1739865892317))

In [63]:
for policy_name in ("POLICY_1","POLICY_2", "POLICY_3"):
    demo_set_policy_on_table(["NS_POLICY"], policy_name, ["NS1","NS2"], "TEST_TABLE_1")

In [67]:

response = collado_client.get_applicable_policies_on_table(prefix=catalog_name, namespace=format_namespace(["NS1","NS2"]), table="TEST_TABLE_1")
response.policies

[Policy(policy_type='test_type_3', name='POLICY_3', description='description', content='demo_content', version=0, created_at_ms=1739865884147, updated_at_ms=1739865884147),
 Policy(policy_type='test_type_2', name='POLICY_2', description='description', content='demo_content', version=0, created_at_ms=1739865884143, updated_at_ms=1739865884143),
 Policy(policy_type='test_type_1', name='POLICY_1', description='new_description', content='new_content', version=0, created_at_ms=1739865884136, updated_at_ms=1739865892317)]

In [57]:
for policy_name in ("POLICY_1","POLICY_2", "POLICY_3"):
    demo_drop_policy(["NS_POLICY"], policy_name)

# Initiate a new Spark session
Change the credentials to the PM's read-only credentials

In [ ]:
# The new spark session inherits everything from the previous session except for the overridden credentials
new_spark = spark.newSession()
new_spark.conf.set("spark.sql.catalog.polaris.credential", f"{reader_principal.credentials.client_id}:{reader_principal.credentials.client_secret}")
new_spark.sql("USE polaris")

# Show Namespace contents
We can still `USE NAMESPACE` and `SHOW TABLES`, which require `READ_NAMESPACE_PROPERTIES` and `LIST_TABLES` privileges respectively

In [ ]:
new_spark.sql("USE NAMESPACE COLLADO_TEST.PUBLIC")
new_spark.sql("SHOW TABLES").show()

# Table reads work

In [ ]:
new_spark.sql("SELECT * FROM TEST_TABLE").show()

# Insert attempts will fail

In [ ]:
new_spark.sql("INSERT INTO TEST_TABLE VALUES (4, 'you cannot see this data'), (5, 'it will never be inserted'), (6, 'sad emoji')")

# Create an API client using reader credentials

In [ ]:
# Create a client to fetch an API token - use the reader's client_id and client_secret as the username/password
token_client = CatalogApiClient(CatalogApiClientConfiguration(username=reader_principal.credentials.client_id,
                                 password=reader_principal.credentials.client_secret,
                                 host='http://polaris:8181/api/catalog'))

# Get the token
pm_token = IcebergOAuth2API(token_client).get_token(scope='PRINCIPAL_ROLE:ALL',
                            client_id=reader_principal.credentials.client_id,
                          client_secret=reader_principal.credentials.client_secret,
                          grant_type='client_credentials',
                          _headers={'realm': 'default-realm'})

# Now create a catalog client that uses the token in its Authentication header
pm_client = IcebergCatalogAPI(CatalogApiClient(CatalogApiClientConfiguration(access_token=pm_token.access_token,
              host='http://polaris:8181/api/catalog')))


# LoadTable returns a similar response
However, the S3 credentials are scoped to read-only

In [ ]:
tbl_meta = pm_client.load_table(prefix=catalog_name, namespace=format_namespace(['COLLADO_TEST', 'PUBLIC']), table='TEST_TABLE', x_iceberg_access_delegation='true')
display(JSON(tbl_meta.to_dict(), expanded=True))

# Metadata manipulation is blocked by Polaris
PMs are always dropping tables in prod

In [ ]:
pm_client.drop_table(prefix=catalog_name, namespace=format_namespace(['COLLADO_TEST', 'PUBLIC']), table='TEST_TABLE')

# Add another Principal Role to the Engineer Principal
A principal can belong to multiple Principal Roles. Typically, a call will use the union of all privilages assigned to all of the principal's roles. 

In [ ]:
# Create a new principal role
ops_role = create_principal_role(root_client, "ops_engineer")

# Grant the read_only catalog role to the new principal role
root_client.assign_catalog_role_to_principal_role(principal_role_name=ops_role.name,
                                                  catalog_name=catalog.name,
                                                  grant_catalog_role_request=GrantCatalogRoleRequest(catalog_role=read_only_role))

# Assign the engineer principal to the new role
# The engineer principal now belongs to _both_ roles
root_client.assign_principal_role(engineer_principal.principal.name, grant_principal_role_request=GrantPrincipalRoleRequest(principal_role=ops_role))

# Scope the spark session to a single role
In this case, the Spark session is down-scoped to only the role specified. Even though the engineer has read-write privileges, the session only has privileges assigned to the specified Principal Role - in this case, the `read_only` catalog role.

In [ ]:
ro_spark = spark.newSession()
ro_spark.conf.set("spark.sql.catalog.polaris.scope", 'PRINCIPAL_ROLE:ops_engineer')
ro_spark.sql("USE polaris")
ro_spark.sql("USE NAMESPACE COLLADO_TEST.PUBLIC")
ro_spark.sql("SHOW TABLES").show()

# The engineer can still read data

In [ ]:
ro_spark.sql("SELECT * FROM TEST_TABLE").show()

# But inserts fail

In [ ]:
ro_spark.sql("INSERT INTO TEST_TABLE VALUES (4, 'you cannot see this data'), (5, 'it will never be inserted'), (6, 'sad emoji')")

# And metadata operations are prohibited
Oops - I didn't mean to drop the _production_ table!

In [ ]:
# create a token client with the _engineer's_ credentials
token_client = CatalogApiClient(CatalogApiClientConfiguration(username=engineer_principal.credentials.client_id,
                                 password=engineer_principal.credentials.client_secret,
                                 host='http://polaris:8181/api/catalog'))

# specify the role I want to activate - only ops_engineer
ops_token = IcebergOAuth2API(token_client).get_token(scope='PRINCIPAL_ROLE:ops_engineer',
                            client_id=engineer_principal.credentials.client_id,
                          client_secret=engineer_principal.credentials.client_secret,
                          grant_type='client_credentials',
                          _headers={'realm': 'default-realm'})

# The returned token is scoped to _only_ the privileges granted to the ops_engineer role
# The ops_client fails to do any real damage even though the engineer normally has DROP_TABLE privileges
ops_client = IcebergCatalogAPI(CatalogApiClient(CatalogApiClientConfiguration(access_token=ops_token.access_token,
              host='http://polaris:8181/api/catalog')))
ops_client.drop_table(prefix=catalog_name, namespace=format_namespace(['COLLADO_TEST', 'PUBLIC']), table='TEST_TABLE')